# LuSIR x4 Colab WebUI

Upload an image and run x4 super-resolution with LuSIR using the public `jwheo/LuSIR` checkpoints.

1. Use **Runtime -> Change runtime type -> GPU**.
2. Run the setup cells below.
3. Use the WebUI upload box, choose options, and press **Run x4 SR**.
4. Compare before/after with the slider and download the SR output.

The default model is the deterministic Residual Refiner v2 path:
`LR -> Stage 2 XL condition encoder -> residual refiner v2 -> Stage 1 VAE decoder`.
It does not run Stage 3 or Stage 4, runs comfortably on T4 for typical images, and remains the recommended public Colab option.


In [ ]:
#@title Prepare repository { display-mode: "form" }
import os
import subprocess
from pathlib import Path

os.chdir('/content')
repo = Path('LuSIR')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/BitIntx/LuSIR.git'], check=True)
os.chdir(repo)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
print('repo:', Path.cwd())


In [ ]:
#@title Install lightweight dependencies { display-mode: "form" }
%pip -q install pyyaml pillow huggingface_hub gradio
%pip -q install -e . --no-deps


In [ ]:
#@title Check GPU { display-mode: "form" }
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required. In Colab, use Runtime -> Change runtime type -> GPU.')
props = torch.cuda.get_device_properties(0)
print('gpu:', torch.cuda.get_device_name(0))
print('gpu memory:', f'{props.total_memory / 1024 ** 3:.1f} GB')


## Launch the WebUI

The LuSIR app downloads the selected checkpoints on first use. Default settings are tuned for user-uploaded LR images. Use the **Residual correction strength** slider to compare softer/conservative versus full correction, and use **Slider left side** to compare SR against bicubic or Stage 2 condition output.


In [ ]:
#@title Launch WebUI { display-mode: "form" }
!python -u tools/demo/colab_webui.py --share


## Notes

- **Residual Refiner v2** is the recommended default and is deterministic.
- Diffusion comparison models are slower and can be softer or more aggressive depending on the input.
- For T4, keep tiling on and start with tile batch size `1` for large images.
- Checkpoints are research prototypes under a non-commercial license.
